<a href="https://colab.research.google.com/github/charlesmrs/CodigoDosAlunos/blob/main/Mini_projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import urllib.request  # biblioteca nativa para baixar arquivos da internet

url_produtos = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/main/olist_products_dataset.csv"
url_pedidos  = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/main/olist_orders_dataset.csv"

urllib.request.urlretrieve(url_produtos, "olist_products_dataset.csv")
urllib.request.urlretrieve(url_pedidos,  "olist_orders_dataset.csv")

print("Arquivos baixados com sucesso!")

Arquivos baixados com sucesso!


In [52]:

import csv
import re
from datetime import datetime

def ler_csv(caminho_arquivo):
    registros = []

    with open(caminho_arquivo, encoding='utf-8') as arquivo:
        leitor = csv.DictReader(arquivo)
        for linha in leitor:
            registros.append(dict(linha))

    return registros


produtos = ler_csv("olist_products_dataset.csv")
pedidos  = ler_csv("olist_orders_dataset.csv")

print(f"Total de produtos carregados : {len(produtos)}")
print(f"Total de pedidos carregados  : {len(pedidos)}")
print()
print("Exemplo — primeiro produto:")
print(produtos[0])
print()
print("Exemplo — primeiro pedido:")
print(pedidos[0])

Total de produtos carregados : 32951
Total de pedidos carregados  : 99441

Exemplo — primeiro produto:
{'product_id': '1e9e8ef04dbcff4541ed26657ea517e5', 'product_category_name': 'perfumaria', 'product_name_lenght': '40', 'product_description_lenght': '287', 'product_photos_qty': '1', 'product_weight_g': '225', 'product_length_cm': '16', 'product_height_cm': '10', 'product_width_cm': '14'}

Exemplo — primeiro pedido:
{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00'}


In [53]:

def tratar_produtos(lista_produtos):

    produtos_limpos     = []
    total_categoria_fix = 0
    total_dimensao_drop = 0


    dimensoes = ['product_weight_g', 'product_length_cm',
                 'product_height_cm', 'product_width_cm']

    for produto in lista_produtos:

        if produto['product_category_name'].strip() == '':
            produto['product_category_name'] = 'Sem Categoria'
            total_categoria_fix += 1


        tem_dimensao_nula = False

        for coluna in dimensoes:
            if produto[coluna].strip() == '':
                tem_dimensao_nula = True
                break

        if tem_dimensao_nula:
            total_dimensao_drop += 1
            continue

        produtos_limpos.append(produto)

    return produtos_limpos, total_categoria_fix, total_dimensao_drop

In [54]:
# ============================================================
# TESTANDO o tratamento de produtos
# ============================================================

produtos_limpos, fix_cat, drop_dim = tratar_produtos(produtos)

print(f"Produtos antes do tratamento  : {len(produtos)}")
print(f"Produtos após o tratamento    : {len(produtos_limpos)}")
print(f"Categorias corrigidas         : {fix_cat}")
print(f"Registros descartados (dim.)  : {drop_dim}")
print()
print("Exemplo produto tratado:")
print(produtos_limpos[0])

Produtos antes do tratamento  : 32951
Produtos após o tratamento    : 32949
Categorias corrigidas         : 610
Registros descartados (dim.)  : 2

Exemplo produto tratado:
{'product_id': '1e9e8ef04dbcff4541ed26657ea517e5', 'product_category_name': 'perfumaria', 'product_name_lenght': '40', 'product_description_lenght': '287', 'product_photos_qty': '1', 'product_weight_g': '225', 'product_length_cm': '16', 'product_height_cm': '10', 'product_width_cm': '14'}


In [55]:
def limpar_categoria(nome):
    """
    Aplica 3 transformações em sequência:
    1. .strip()  → remove espaços no início e no fim
    2. .lower()  → converte tudo para minúsculas
    3. re.sub()  → remove caracteres especiais (só aceita letras e _)
    """

    nome = nome.strip()
    nome = nome.lower()
    nome = re.sub(r'[^a-z0-9_\s]', '', nome)
    nome = re.sub(r'\s+', ' ', nome).strip()

    return nome


def padronizar_categorias(lista_produtos):
    for produto in lista_produtos:
        produto['product_category_name'] = limpar_categoria(
            produto['product_category_name']
        )
    return lista_produtos

In [56]:
produtos_limpos = padronizar_categorias(produtos_limpos)

categorias_unicas = set(p['product_category_name'] for p in produtos_limpos)

print(f"Total de categorias únicas: {len(categorias_unicas)}")
print()
print("Primeiras 10 categorias padronizadas:")
for cat in sorted(list(categorias_unicas))[:10]:
    print(f"  → {cat}")

Total de categorias únicas: 74

Primeiras 10 categorias padronizadas:
  → agro_industria_e_comercio
  → alimentos
  → alimentos_bebidas
  → artes
  → artes_e_artesanato
  → artigos_de_festas
  → artigos_de_natal
  → audio
  → automotivo
  → bebes


In [57]:

def processar_pedidos(lista_pedidos):

    pedidos_processados  = []
    pedidos_cancelados   = []
    total_cancelados     = 0

    for pedido in lista_pedidos:

        data_entrega = pedido['order_delivered_customer_date'].strip()

        if data_entrega == '':
            if pedido['order_status'] == 'canceled':
                pedidos_cancelados.append(pedido)
                total_cancelados += 1


        data_aprovacao = pedido['order_approved_at'].strip()

        if data_aprovacao != '':
            data_obj = datetime.strptime(data_aprovacao, '%Y-%m-%d %H:%M:%S')
            pedido['order_approved_at'] = data_obj.strftime('%d/%m/%Y')
        else:
            pedido['order_approved_at'] = 'Sem data de aprovação'

        pedidos_processados.append(pedido)

    return pedidos_processados, pedidos_cancelados, total_cancelados

In [58]:
# ============================================================
# TESTANDO o processamento de pedidos
# ============================================================

pedidos_processados, pedidos_cancelados, total_cancelados = processar_pedidos(pedidos)

print(f"Total de pedidos processados : {len(pedidos_processados)}")
print(f"Total de pedidos cancelados  : {total_cancelados}")
print()

print("Verificando hipótese: datas vazias = pedidos cancelados?")
outros_status = [p for p in pedidos_processados
                 if p['order_delivered_customer_date'].strip() == ''
                 and p['order_status'] != 'canceled']
print(f"  Pedidos sem data de entrega E não cancelados: {len(outros_status)}")
print(f"  Pedidos cancelados confirmados              : {total_cancelados}")
print()

print("Exemplo de data convertida:")
exemplo = next(p for p in pedidos_processados if p['order_approved_at'] != 'Sem data de aprovação')
print(f"  order_approved_at → {exemplo['order_approved_at']}")

Total de pedidos processados : 99441
Total de pedidos cancelados  : 619

Verificando hipótese: datas vazias = pedidos cancelados?
  Pedidos sem data de entrega E não cancelados: 2346
  Pedidos cancelados confirmados              : 619

Exemplo de data convertida:
  order_approved_at → 02/10/2017


In [59]:

def exibir_relatorio(produtos_orig, produtos_limpos,
                     fix_cat, drop_dim,
                     total_pedidos, total_cancelados,
                     outros_sem_entrega):

    print("=" * 55)
    print("       RELATÓRIO DE SANITIZAÇÃO — OLIST ETL")
    print("=" * 55)

    print("\n PRODUTOS")
    print(f"  Total lido no CSV              : {produtos_orig}")
    print(f"  Registros descartados (dim.)   : {drop_dim}")
    print(f"  Registros válidos              : {len(produtos_limpos)}")
    print(f"  Categorias corrigidas (nulo)   : {fix_cat}")

    print("\n PEDIDOS")
    print(f"  Total lido no CSV              : {total_pedidos}")
    print(f"  Pedidos cancelados             : {total_cancelados}")
    print(f"  Sem entrega (outros status)    : {outros_sem_entrega}")

    print("\n HIPÓTESE DE NEGÓCIO")
    print("  Pedidos cancelados SEMPRE têm data de entrega vazia?")
    print("  → VERDADEIRO para cancelados.")
    print("  → Mas existem 2346 pedidos sem entrega com outros")
    print("    status (ex: shipped, processing). Base NÃO está")
    print("    completamente sanitizada nesse ponto.")

    print("\n" + "=" * 55)
    print("  Pipeline concluído com sucesso!")
    print("=" * 55)

In [60]:
# ============================================================
# MAIN — Pipeline completo de sanitização
# ============================================================

def main():

    print("🚀 Iniciando pipeline de sanitização Olist...\n")

    # ETAPA 1 — Leitura dos arquivos
    produtos = ler_csv("olist_products_dataset.csv")
    pedidos  = ler_csv("olist_orders_dataset.csv")
    print(f"✅ CSVs lidos — {len(produtos)} produtos | {len(pedidos)} pedidos")

    # ETAPA 2 — Tratamento de nulos nos produtos
    produtos_limpos, fix_cat, drop_dim = tratar_produtos(produtos)
    print(f"✅ Nulos tratados — {drop_dim} descartados | {fix_cat} categorias corrigidas")

    # ETAPA 3 — Padronização de strings com Regex
    produtos_limpos = padronizar_categorias(produtos_limpos)
    print(f"✅ Categorias padronizadas")

    # ETAPA 4 — Regras de negócio + conversão de datas
    pedidos_proc, pedidos_cancelados, total_cancel = processar_pedidos(pedidos)
    outros_sem_entrega = len([
        p for p in pedidos_proc
        if p['order_delivered_customer_date'].strip() == ''
        and p['order_status'] != 'canceled'
    ])
    print(f"✅ Pedidos processados — {total_cancel} cancelados identificados")

    # ETAPA 5 — Relatório final
    print()
    exibir_relatorio(
        len(produtos), produtos_limpos,
        fix_cat, drop_dim,
        len(pedidos_proc), total_cancel,
        outros_sem_entrega
    )

# Executa o pipeline
main()

🚀 Iniciando pipeline de sanitização Olist...

✅ CSVs lidos — 32951 produtos | 99441 pedidos
✅ Nulos tratados — 2 descartados | 610 categorias corrigidas
✅ Categorias padronizadas
✅ Pedidos processados — 619 cancelados identificados

       RELATÓRIO DE SANITIZAÇÃO — OLIST ETL

 PRODUTOS
  Total lido no CSV              : 32951
  Registros descartados (dim.)   : 2
  Registros válidos              : 32949
  Categorias corrigidas (nulo)   : 610

 PEDIDOS
  Total lido no CSV              : 99441
  Pedidos cancelados             : 619
  Sem entrega (outros status)    : 2346

 HIPÓTESE DE NEGÓCIO
  Pedidos cancelados SEMPRE têm data de entrega vazia?
  → VERDADEIRO para cancelados.
  → Mas existem 2346 pedidos sem entrega com outros
    status (ex: shipped, processing). Base NÃO está
    completamente sanitizada nesse ponto.

  Pipeline concluído com sucesso!


In [61]:
readme_content = """
 Mini-Projeto Avaliativo — Pipeline de Sanitização Olist

 Identificação

 Aluno:Charles Moraes Rodrigues
 Curso:Machine Learning e Visão Computacional
 Turma:T1
 Entrega: Mini-Projeto Avaliativo
 Módulo Módulo 1 — Semana 07
 Instituição SCTEC — Santa Catarina


Descrição do Projeto

A Olist, empresa brasileira de e-commerce, enfrenta um desafio
frequente no mundo real: arquivos CSV extraídos do banco de dados chegam
com inconsistências — categorias de produtos vazias, dimensões físicas
nulas, datas em formato inadequado e pedidos cancelados misturados com
pedidos válidos.

Este projeto implementa um pipeline de ETL (Extract, Transform, Load)
em Python puro, utilizando apenas bibliotecas nativas (`csv`, `re`,
`datetime`), sem o uso do Pandas. O objetivo é sanitizar dois datasets
oficiais da Olist antes que os dados sejam usados em modelos de
Machine Learning ou relatórios de Business Intelligence.


 Estrutura do Projeto

mini_projeto_olist/
├── main.ipynb                    ← Notebook principal com todo o pipeline
├── README.md                     ← Este arquivo de documentação
├── olist_orders_dataset.csv      ← Dataset de pedidos da Olist
└── olist_products_dataset.csv    ← Dataset de produtos da Olist


   Explicação das Funções

  `ler_csv(caminho_arquivo)`
Responsável por abrir e ler qualquer arquivo CSV usando o `csv.DictReader`.
Cada linha do arquivo é convertida em um dicionário Python onde as chaves
são os nomes das colunas do cabeçalho. Retorna uma lista com todos os
registros carregados em memória.

  python
  Exemplo de uso
produtos = ler_csv("olist_products_dataset.csv")
 Retorna: [{'product_id': '...', 'product_category_name': 'perfumaria', ...}, ...]

 tratar_produtos(lista_produtos)
Percorre todos os produtos aplicando duas regras de validação:

- Regra 1 — Categoria vazia: Se o campo `product_category_name
  estiver vazio, é preenchido com a string "Sem Categoria".

- Regra 2 — Dimensões físicas nulas: Se qualquer campo de dimensão
  (`product_weight_g`, `product_length_cm`, `product_height_cm`,
  `product_width_cm`) estiver vazio, o registro é **descartado.

  Justificativa técnica:As dimensões físicas são essenciais para
  cálculo de frete. Imputar valores falsos (como média) causaria
  distorções no modelo de previsão logística. O descarte é a escolha
  mais segura para garantir integridade dos dados.

  python
produtos_limpos, fix_cat, drop_dim = tratar_produtos(produtos)
Resultado: 32949 válidos - 2 descartados - 610 categorias corrigidas


limpar_categoria(nome) e padronizar_categorias(lista_produtos)
Aplica três transformações em sequência em cada nome de categoria:

1. .strip() — Remove espaços em branco no início e no fim
2. .lower() — Converte todo o texto para letras minúsculas
3. re.sub() — Remove caracteres especiais com Expressão Regular,
   mantendo apenas letras, números, underline e espaços

  python
  Exemplo
limpar_categoria("  Eletrônicos!! ") → "eletrnicos"
limpar_categoria("cama_mesa_banho")  → "cama_mesa_banho"

  `processar_pedidos(lista_pedidos)`
Processa cada pedido realizando duas operações:

- Regra de Negócio: Identifica pedidos com
  `order_delivered_customer_date` vazia e verifica se o status é
  `canceled`, separando-os em uma lista exclusiva para validar a
  hipótese de negócio da Olist.

- Conversão de Data:Lê o campo `order_approved_at` no formato
  original "2017-10-02 11:07:15" usando `datetime.strptime()` e
  converte para o formato brasileiro "02/10/2017" com `strftime()`.

  python
pedidos_proc, cancelados, total = processar_pedidos(pedidos)
  Resultado: 619 cancelados identificados
`exibir_relatorio(...)`
Ao final do pipeline, exibe na tela um sumário estatístico completo
com todos os indicadores do processamento: total de registros lidos,
corrigidos, descartados e pedidos cancelados identificados.

  Resultado Final do Pipeline
=======================================================
       RELATÓRIO DE SANITIZAÇÃO — OLIST ETL
=======================================================

    PRODUTOS
  Total lido no CSV              : 32.951
  Registros descartados (dim.)   : 2
  Registros válidos              : 32.949
  Categorias corrigidas (nulo)   : 610

    PEDIDOS
  Total lido no CSV              : 99.441
  Pedidos cancelados             : 619
  Sem entrega (outros status)    : 2.346

   HIPÓTESE DE NEGÓCIO
  Pedidos cancelados SEMPRE têm data de entrega vazia?
  → VERDADEIRO para cancelados.
  → Mas existem 2.346 pedidos sem entrega com outros
    status (ex: shipped, processing).


  Reflexão Teórica — Qualidade de Dados e Machine Learning

A limpeza rigorosa de dados é a base de qualquer modelo de Machine
Learning confiável. Quando alimentamos um algoritmo com registros nulos,
categorias inconsistentes ou datas mal formatadas, induzimos o modelo a
aprender padrões falsos — fenômeno conhecido como Garbage In, Garbage
Out. No contexto de Overfitting, dados sujos com ruídos e
inconsistências fazem o modelo "decorar" anomalias ao invés de aprender
padrões reais, prejudicando sua capacidade de generalização em dados novos.
Já o viés (bias) é introduzido quando categorias despadronizadas como
"Perfumaria" e "perfumaria" são tratadas como classes distintas,
distorcendo completamente a distribuição que o modelo aprende.

No nosso pipeline, cada decisão técnica foi orientada a evitar esses
problemas: categorias padronizadas garantem que o modelo não confunda
classes idênticas escritas de formas diferentes; o descarte de produtos
sem dimensões físicas evita que o modelo de previsão de frete aprenda com
valores imputados artificialmente; e a separação de pedidos cancelados
impede que o algoritmo associe erroneamente ausência de data de entrega
com entregas realizadas. Dados limpos produzem modelos honestos — e
modelos honestos produzem decisões de negócio confiáveis e seguras.



 Tecnologias Utilizadas

 Biblioteca | Uso no Projeto |

csv -Leitura estruturada dos arquivos CSV com `DictReader`.
re  -Expressões Regulares para limpeza de strings.
datetime -  Conversão e formatação de datas.
urllib.request - Download automático dos CSVs no Colab.


Projeto desenvolvido como parte do Mini-Projeto Avaliativo do Módulo 1.
SCTEC — Santa Catarina - Machine Learning e Visão Computacional T1.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Arquivo README.md criado com sucesso!")

Arquivo README.md criado com sucesso!


In [62]:
print(readme_content)


 Mini-Projeto Avaliativo — Pipeline de Sanitização Olist

 Identificação

 Aluno:Charles Moraes Rodrigues
 Curso:Machine Learning e Visão Computacional
 Turma:T1
 Entrega: Mini-Projeto Avaliativo
 Módulo Módulo 1 — Semana 07
 Instituição SCTEC — Santa Catarina


Descrição do Projeto

A Olist, empresa brasileira de e-commerce, enfrenta um desafio
frequente no mundo real: arquivos CSV extraídos do banco de dados chegam
com inconsistências — categorias de produtos vazias, dimensões físicas
nulas, datas em formato inadequado e pedidos cancelados misturados com
pedidos válidos.

Este projeto implementa um pipeline de ETL (Extract, Transform, Load)
em Python puro, utilizando apenas bibliotecas nativas (`csv`, `re`,
`datetime`), sem o uso do Pandas. O objetivo é sanitizar dois datasets
oficiais da Olist antes que os dados sejam usados em modelos de
Machine Learning ou relatórios de Business Intelligence.


 Estrutura do Projeto

mini_projeto_olist/
├── main.ipynb                    ← Noteboo